# TFM Tenerife — Tarea 2.1: Sentimiento general por reseña (BERT Multilingüe)

Analiza el sentimiento de las reseñas de TripAdvisor y Booking (2022 en adelante) con `nlptown/bert-base-multilingual-uncased-sentiment`, las cruza espacialmente con la malla H3, y sube el resultado a `gold.nlp_sentimiento_resenas`.

**Sobre el modelo:** no hace falta descargarlo a mano — la celda del Paso 3 lo descarga solo la primera vez que se ejecuta, directamente desde Hugging Face.

**Diseño resistente a cortes:** el resultado se va guardando en `gold` por bloques (cada ~320 reseñas), no todo junto al final. Si el proceso se interrumpe (se va la luz, se cierra el portátil...), solo se pierde el bloque que estaba a medias — vuelve a ejecutar el notebook entero y, gracias al filtro incremental del Paso 4, continuará justo donde se quedó, sin repetir ni duplicar nada.

## Paso 0 -- Setup de Colab (secrets + GPU)

Version adaptada para Google Colab del notebook original de Guille (`analytics/tarea2/nlp_sentimiento_2_1.ipynb`). Reemplaza el `.env` local por Colab secrets -- Colab no tiene filesystem persistente para un `.env`.

Antes de ejecutar: icono de llave (🔑) en el panel izquierdo de Colab -> agregar los 4 secrets `AZURE_DB_HOST`, `AZURE_DB_USER`, `AZURE_DB_PASSWORD`, `AZURE_DB_NAME` (mismos valores que el `.env` del repo) y activar el acceso del notebook para cada uno.

Tambien conviene (no obligatorio): Runtime -> Change runtime type -> GPU. Sin GPU este notebook funciona igual pero el Paso 6 puede tardar varias horas (ver estimacion original en las Notas al final).

In [ ]:
# Instalacion puntual de dependencias (no requirements.txt completo -- mismo
# criterio que ya usa el equipo en la VM del proyecto para no arrastrar
# dependencias innecesarias). python-dotenv no hace falta en Colab.
!pip install -q transformers torch sentencepiece sqlalchemy psycopg2-binary pandas

import torch
print('GPU disponible:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('AVISO: no hay GPU asignada en este runtime -- Runtime > Change runtime type > GPU.')
    print('Sin GPU, el Paso 6 puede tardar varias horas (55k+ resenas en CPU).')


## Paso 1 — Instalar dependencias

In [ ]:
# (Cubierto por la celda de Setup de Colab de arriba -- se deja esta celda
# solo para no romper la numeracion de pasos del notebook original.)
print('Dependencias ya instaladas en el Paso 0.')

## Paso 2 — Conectar

In [ ]:
from google.colab import userdata
from sqlalchemy import create_engine, text
import pandas as pd

AZURE_DB_HOST = userdata.get('AZURE_DB_HOST')
AZURE_DB_USER = userdata.get('AZURE_DB_USER')
AZURE_DB_PASSWORD = userdata.get('AZURE_DB_PASSWORD')
AZURE_DB_NAME = userdata.get('AZURE_DB_NAME')
AZURE_DB_PORT = 5432  # fijo, no hay secret separado para esto (igual que en analytics/mgwr/scripts/_db.py)

db_url = f'postgresql+psycopg2://{AZURE_DB_USER}:{AZURE_DB_PASSWORD}@{AZURE_DB_HOST}:{AZURE_DB_PORT}/{AZURE_DB_NAME}'
engine = create_engine(db_url, pool_pre_ping=True, pool_recycle=280, connect_args={'sslmode': 'require'})

with engine.connect() as conn:
    version = conn.execute(text('SELECT postgis_version();')).scalar()
print('Conectado. PostGIS:', version)


## Paso 3 — Cargar el modelo (se descarga solo, la primera vez)

In [ ]:
import torch
from transformers import pipeline

dispositivo = 0 if torch.cuda.is_available() else -1  # 0 = GPU, -1 = CPU
print('Usando', 'GPU' if dispositivo == 0 else 'CPU (sera mas lento)')

sentiment_pipe = pipeline(
    'text-classification',
    model='nlptown/bert-base-multilingual-uncased-sentiment',
    device=dispositivo,
)
print('Modelo cargado.')

## Paso 4 — Extraer reseñas NUEVAS de TripAdvisor + Booking (2022 en adelante)

Se unen las dos fuentes y se excluye lo que ya tenga resultado en `gold.nlp_sentimiento_resenas` -- así no hace falta esperar a que la recopilación de datos esté completa: cada vez que se relance, solo procesa lo que sea nuevo desde la última vez, sin duplicar ni repetir trabajo ya hecho.

In [ ]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE TABLE IF NOT EXISTS gold.nlp_sentimiento_resenas (
            resena_id text,
            hotel_id text,
            score integer,
            h3_index text,
            fuente text
        )
    '''))
print('Tabla gold.nlp_sentimiento_resenas lista (creada si no existia).')

In [ ]:
consulta_resenas = '''
    SELECT 'tripadvisor' AS fuente, r.review_id::text AS resena_id, r.location_id::text AS hotel_id, r.texto AS review_text
    FROM silver.tripadvisor_resenas r
    WHERE r.texto IS NOT NULL AND EXTRACT(YEAR FROM r.fecha_publicacion) > 2021
      AND NOT EXISTS (
          SELECT 1 FROM gold.nlp_sentimiento_resenas g
          WHERE g.resena_id = r.review_id::text AND g.fuente = 'tripadvisor'
      )

    UNION ALL

    SELECT 'booking' AS fuente, b.review_id AS resena_id, b.establishment_id AS hotel_id, b.review_text
    FROM silver.silver_booking_reviews b
    WHERE b.review_text IS NOT NULL AND EXTRACT(YEAR FROM b.review_date) > 2021
      AND NOT EXISTS (
          SELECT 1 FROM gold.nlp_sentimiento_resenas g
          WHERE g.resena_id = b.review_id AND g.fuente = 'booking'
      );
'''

df_resenas = pd.read_sql(consulta_resenas, engine)
print('Reseñas NUEVAS a analizar esta vez:', len(df_resenas))
print(df_resenas['fuente'].value_counts())

## Paso 5 — Cruce espacial con la malla H3 (una sola vez, antes de procesar)

TripAdvisor ya tiene geometría (`tripadvisor_ubicaciones.geometry`, EPSG:32628). Booking solo tiene `latitude`/`longitude`, así que se construye la geometría al vuelo y se reproyecta para que coincida con la malla.

In [ ]:
with engine.connect() as conn:
    srid_h3 = conn.execute(text('SELECT ST_SRID(geometry) FROM silver.silver_h3_grid LIMIT 1;')).scalar()
print('SRID de la malla H3:', srid_h3)

consulta_cruce_h3 = f'''
    WITH establecimientos_geo AS (
        SELECT location_id::text AS hotel_id, ST_Transform(geometry, {srid_h3}) AS geometry
        FROM silver.tripadvisor_ubicaciones

        UNION ALL

        SELECT establishment_id AS hotel_id,
               ST_Transform(ST_SetSRID(ST_MakePoint(longitude, latitude), 4326), {srid_h3}) AS geometry
        FROM silver.silver_booking_establishments
        WHERE latitude IS NOT NULL AND longitude IS NOT NULL
    )
    SELECT e.hotel_id, h.h3_index
    FROM establecimientos_geo e
    JOIN silver.silver_h3_grid h ON ST_Contains(h.geometry, e.geometry);
'''

df_hotel_h3 = pd.read_sql(consulta_cruce_h3, engine)
print('Establecimientos con hexagono asignado:', len(df_hotel_h3))

## Paso 6 — Procesar y guardar por bloques (resistente a cortes)

Cada bloque de ~320 reseñas (10 lotes de 32) se analiza, se cruza con su hexágono, y se sube a `gold` **inmediatamente** -- no se espera al final. Imprime el ritmo real y una estimación del tiempo que falta, para que puedas decidir sobre la marcha si esperar, dejarlo corriendo de noche, o buscar acceso a GPU.

In [ ]:
import time

TAMANO_LOTE = 32
FILAS_POR_CHECKPOINT = 320  # se guarda en gold cada ~10 lotes

total = len(df_resenas)
procesadas_total = 0
inicio_general = time.time()

if total == 0:
    print('No hay resenas nuevas que procesar -- todo lo disponible ya esta en gold.')
else:
    for inicio_chunk in range(0, total, FILAS_POR_CHECKPOINT):
        chunk = df_resenas.iloc[inicio_chunk:inicio_chunk + FILAS_POR_CHECKPOINT].copy()
        lista_textos = chunk['review_text'].tolist()

        resultados_chunk = []
        for i in range(0, len(lista_textos), TAMANO_LOTE):
            lote = lista_textos[i:i + TAMANO_LOTE]
            resultados_lote = sentiment_pipe(lote, batch_size=TAMANO_LOTE, truncation=True, max_length=512)
            resultados_chunk.extend(resultados_lote)

        chunk['score'] = [int(r['label'].split()[0]) for r in resultados_chunk]
        chunk = chunk.merge(df_hotel_h3, on='hotel_id', how='left')

        resultado_chunk = chunk[['resena_id', 'hotel_id', 'score', 'h3_index', 'fuente']]
        resultado_chunk.to_sql('nlp_sentimiento_resenas', engine, schema='gold', if_exists='append', index=False)

        procesadas_total += len(chunk)
        transcurrido = time.time() - inicio_general
        velocidad = procesadas_total / transcurrido if transcurrido > 0 else 0
        restantes = total - procesadas_total
        eta_min = (restantes / velocidad / 60) if velocidad > 0 else 0

        print(f'Guardadas {procesadas_total}/{total} -- {velocidad:.1f} resenas/seg -- estimado restante: {eta_min:.0f} min')

    with engine.begin() as conn:
        conn.execute(text(
            'CREATE INDEX IF NOT EXISTS idx_nlp_sentimiento_hotel_id '
            'ON gold.nlp_sentimiento_resenas (hotel_id)'
        ))
        conn.execute(text(
            'CREATE INDEX IF NOT EXISTS idx_nlp_sentimiento_h3_index '
            'ON gold.nlp_sentimiento_resenas (h3_index)'
        ))

    print()
    print('Completado. Total procesado y guardado en esta ejecucion:', procesadas_total)

## Paso 7 — Verificar

In [ ]:
with engine.connect() as conn:
    resumen = pd.read_sql('''
        SELECT fuente, COUNT(*) AS total, AVG(score) AS score_medio,
               COUNT(h3_index) AS con_hexagono
        FROM gold.nlp_sentimiento_resenas
        GROUP BY fuente
    ''', conn)

display(resumen)

## Notas

- El ticket original solo menciona `resena_id`, `hotel_id`, `score` como columnas de salida. He añadido `h3_index` (para que el cruce espacial sirva de algo en la propia tabla) y `fuente` (para poder distinguir TripAdvisor de Booking después). Si el equipo prefiere la tabla más ajustada al ticket exacto, es fácil quitar estas dos columnas antes de subir.
- `hotel_id` incluye tanto hoteles como restaurantes de TripAdvisor (mismo nombre de columna que usa el ticket, aunque el contenido no sea solo hoteles).
- **Diseño incremental y resistente a cortes**: el Paso 4 excluye lo que ya tenga resultado en `gold`, y el Paso 6 guarda cada bloque de ~320 reseñas según lo va terminando. Si el proceso se corta a mitad, solo se pierde el bloque en curso -- vuelve a ejecutar el notebook entero (Pasos 1-7) y continuará justo donde se quedó, sin duplicar nada.
- No he podido ejecutar esto contra los datos ni el modelo reales. Si algo falla (nombre de columna, memoria insuficiente, etc.), pégame el error exacto.
- Con 55.787 reseñas de Booking (2022 en adelante) más las de TripAdvisor, en CPU esto puede tardar varias horas. El Paso 6 imprime el ritmo real y una estimación de tiempo restante nada más arrancar -- si ves que va a tardar demasiado, puedes interrumpirlo con tranquilidad (Kernel → Interrupt) y retomarlo más tarde sin perder el progreso ya guardado.

---

**Nota de adaptacion a Colab:** esta es una copia adaptada del notebook original de Guille (`analytics/tarea2/nlp_sentimiento_2_1.ipynb`), pensada para correr en Google Colab en paralelo al trabajo del Bloque 5 en otra maquina. El original queda intacto como referencia. Cambios respecto al original: credenciales via Colab secrets en vez de `.env`, celda de Setup de Colab (Paso 0) agregada al principio. La logica de negocio (queries, modelo, cruce espacial) no se modifico.